# 04 – Iteration 3: Dataset Preparation

This notebook:

1. Loads the feature table and selected features list  
2. Removes leakage features (anomaly codes and flags)  
3. Performs **group-aware** train/valid/test splitting  
4. Fits preprocessing (Median Imputer + Standard Scaler) on training set only  
5. Applies preprocessing to all splits  
6. Saves all prepared matrices + preprocessing params for model training  


Imports & Paths

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Allow importing from src/
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from splitting import make_group_splits, add_split_column
from preprocessing import fit_preprocessor, apply_preprocessor, save_preprocessor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

FEATURES_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "features"))
SELECTED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "selected"))
PREPARED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "prepared"))
os.makedirs(PREPARED_DIR, exist_ok=True)

FEATURES_PARQUET_PATH = os.path.join(FEATURES_DIR, "features_dataset2_iter3.parquet")
SELECTED_FEATURES_PATH = os.path.join(SELECTED_DIR, "selected_features_iter3.txt")

print("FEATURES:", FEATURES_PARQUET_PATH)
print("SELECTED FEATURES:", SELECTED_FEATURES_PATH)
print("OUTPUT DIR:", PREPARED_DIR)


FEATURES: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features\features_dataset2_iter3.parquet
SELECTED FEATURES: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\selected\selected_features_iter3.txt
OUTPUT DIR: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared


Load Data & Selected Features

In [2]:
# 1) Load feature table
feat_df = pd.read_parquet(FEATURES_PARQUET_PATH)

print("Loaded features. Shape:", feat_df.shape)

# 2) Load selected feature names
with open(SELECTED_FEATURES_PATH, "r") as f:
    selected_features = [line.strip() for line in f]

print("Feature list from selection:", selected_features)


Loaded features. Shape: (21195970, 19)
Feature list from selection: ['CODI_ANOMALIA', 'period_hours', 'meter_std', 'meter_mean', 'flag_anom_163840', 'cons_z_meter', 'flag_anom_32768', 'CONSUMO_REAL', 'cons_lag1', 'delta1']


Define ID/LABEL & Remove Leakage Columns

In [3]:
ID_COL = "NUMEROSERIECONTADOR"
LABEL_COL = "y_anom"

LEAKAGE_COLS = [
    "CODI_ANOMALIA",
    "flag_anom_32768",
    "flag_anom_163840",
]

# Remove leaking features
feat_df = feat_df.drop(columns=[c for c in LEAKAGE_COLS if c in feat_df.columns])

# Ensure none of the leakage features are in selected features
selected_features = [f for f in selected_features if f not in LEAKAGE_COLS]

print("Final selected features:", selected_features)


Final selected features: ['period_hours', 'meter_std', 'meter_mean', 'cons_z_meter', 'CONSUMO_REAL', 'cons_lag1', 'delta1']


Create Group-Aware Splits

In [4]:
train_idx, valid_idx, test_idx = make_group_splits(
    df=feat_df,
    id_col=ID_COL,
    train_size=0.70,
    valid_size=0.15,
    random_state=42,
)

feat_df = add_split_column(feat_df, train_idx, valid_idx, test_idx)

feat_df["split"].value_counts()


split
train    14811416
test      3202540
valid     3182014
Name: count, dtype: int64

Split into Train/Valid/Test Frames

In [5]:
df_train = feat_df[feat_df["split"] == "train"]
df_valid = feat_df[feat_df["split"] == "valid"]
df_test  = feat_df[feat_df["split"] == "test"]

print("Train shape:", df_train.shape)
print("Valid shape:", df_valid.shape)
print("Test  shape:", df_test.shape)


Train shape: (14811416, 17)
Valid shape: (3182014, 17)
Test  shape: (3202540, 17)


Fit Preprocessor (Median + StandardScaler)

In [6]:
imputer, scaler = fit_preprocessor(
    df=df_train,
    feature_cols=selected_features,
)

print("Preprocessor fitted.")


MemoryError: Unable to allocate 791. MiB for an array with shape (7, 14811416) and data type float64

Apply Preprocessor

In [ ]:
X_train = apply_preprocessor(df_train, selected_features, imputer, scaler)
X_valid = apply_preprocessor(df_valid, selected_features, imputer, scaler)
X_test  = apply_preprocessor(df_test, selected_features, imputer, scaler)

y_train = df_train[LABEL_COL].values
y_valid = df_valid[LABEL_COL].values
y_test  = df_test[LABEL_COL].values

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)


Save Prepared Data

In [ ]:
np.save(os.path.join(PREPARED_DIR, "X_train.npy"), X_train)
np.save(os.path.join(PREPARED_DIR, "X_valid.npy"), X_valid)
np.save(os.path.join(PREPARED_DIR, "X_test.npy"), X_test)

np.save(os.path.join(PREPARED_DIR, "y_train.npy"), y_train)
np.save(os.path.join(PREPARED_DIR, "y_valid.npy"), y_valid)
np.save(os.path.join(PREPARED_DIR, "y_test.npy"), y_test)

print("[OK] Saved all split matrices.")


Save Preprocessor Parameters

In [ ]:
save_preprocessor(
    imputer=imputer,
    scaler=scaler,
    feature_cols=selected_features,
    out_dir=PREPARED_DIR,
    base_name="preprocessor_iter3",
)

print("Preprocessor parameters saved.")


Summary for Documentation

In [ ]:
print("Selected features:", selected_features)
print("Train rows:", X_train.shape[0])
print("Valid rows:", X_valid.shape[0])
print("Test rows:", X_test.shape[0])
